# Distributed Training with DDP

In this lab, we spread a model's training over several processes with `DistributedDataParallel` (DDP), PyTorch's standard mechanism for **data parallelism**.

Colab only provides one GPU: we will therefore launch 2 processes **on CPU**, communicating through the `gloo` backend. The code is exactly what you would use on several GPUs, except for the backend (`nccl`): what you write here works as is on a machine with 4 or 8 GPUs.

Outline:

1. write a DDP training script and launch it with `torchrun`;
2. check that the processes stay synchronized;
3. save a checkpoint correctly;
4. do the same in a few lines with PyTorch Lightning.

In [ ]:
!pip install -q lightning

## The starting point

Here is a classic, single-process training script for a synthetic classification problem. The `%%writefile` magic writes the cell's content to a file instead of running it.

In [ ]:
%%writefile common.py
import torch
from torch import nn
from torch.utils.data import TensorDataset


def make_dataset(n: int = 8192) -> TensorDataset:
  # Same seed in every process: they all see the same dataset
  generator = torch.Generator().manual_seed(0)
  x = torch.randn(n, 20, generator=generator)
  true_weights = torch.randn(20, 3, generator=generator)
  y = (x @ true_weights).argmax(dim=1)
  return TensorDataset(x, y)


def make_model() -> nn.Module:
  torch.manual_seed(0)
  return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 3))

In [ ]:
%%writefile train_single.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model

dataset = make_dataset()
model = make_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

for epoch in range(3):
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch}: {len(loader)} batches, last loss {loss.item():.3f}")

In [ ]:
!python train_single.py

## A DDP script

With DDP, `torchrun` launches several **processes** running the same script. Each process:

- joins the process group (`dist.init_process_group`); `torchrun` gives it its number (*rank*) and the total number of processes through environment variables;
- wraps its model in `DistributedDataParallel`, which averages gradients across processes during `backward()`;
- only reads its share of the dataset, thanks to a `DistributedSampler`.

*Complete the `train_ddp.py` script below (the `...`):*

- *initialize the process group with the `"gloo"` backend and get the rank and number of processes (`dist.get_rank()`, `dist.get_world_size()`);*
- *wrap the model in `DDP`;*
- *create a `DistributedSampler` and pass it to the `DataLoader` (instead of `shuffle=True`);*
- *call `sampler.set_epoch(epoch)` at the start of each epoch;*
- *only print messages from the rank 0 process.*

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

...  # Process group initialization, rank and number of processes

dataset = make_dataset()
model = ...  # Model wrapped in DDP
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = ...
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  ...
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  ...  # Print from rank 0 only

dist.destroy_process_group()

### Solution

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")  # "nccl" on GPU
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())  # on GPU: DDP(make_model().to(rank), device_ids=[rank])
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if rank == 0:
    print(f"Epoch {epoch}: {len(loader)} batches per process, "
          f"{world_size} processes, last loss {loss.item():.3f}")

dist.destroy_process_group()

Launch the script on 2 processes:

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

*Answer the following questions:*

1. *How many batches does each process handle per epoch, compared with the single-process script? Why?*
2. *Each process uses batches of 64 examples. How many examples contribute to each weight update? What does it imply for the choice of learning rate?*
3. *What is `sampler.set_epoch(epoch)` for?*

*Your answers here.*

### Solution

1. Each process handles 64 batches per epoch instead of 128: the `DistributedSampler` gives each process a different half of the dataset. With 2 GPUs, an epoch therefore takes about half the time.
2. Gradients are averaged across the 2 processes: each update uses 2 × 64 = 128 examples. The effective batch grows with the number of processes; the learning rate is often increased accordingly (with a *warmup*), otherwise training progresses less per epoch.
3. The `DistributedSampler` shuffles the data with a seed that depends on the epoch: without `set_epoch`, every epoch would see the examples in the same order.

## Do the processes stay synchronized?

Each process has its own copy of the model. DDP guarantees they stay identical: weights are broadcast from rank 0 when the `DDP` is created, then all processes apply the same update, computed from the averaged gradients.

*Let's check it. In a copy of the script (`train_ddp_check.py`), after training, compute in each process the sum of all the model's weights, then gather these sums with `dist.all_gather` and print them from rank 0.*

In [ ]:
# Your code here

### Solution

In [ ]:
%%writefile train_ddp_check.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    cross_entropy(model(x), y).backward()
    optimizer.step()

checksum = torch.stack([p.detach().sum() for p in model.parameters()]).sum()
checksums = [torch.zeros(()) for _ in range(world_size)]
dist.all_gather(checksums, checksum)
if rank == 0:
  print("Sum of the weights in each process:", [f"{c.item():.6f}" for c in checksums])

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp_check.py

## Saving a checkpoint

In distributed training, all processes have the same weights: only one must write the checkpoint, otherwise they all write the same file at the same time.

*Modify `train_ddp.py` to save the model at the end of training, from rank 0 only, to `model.pt`. Then check, in a notebook cell, that you can reload these weights into a **non-distributed** model created with `make_model()`.*

*Hint: the original model is available as `model.module`. Why save `model.module.state_dict()` rather than `model.state_dict()`?*

In [ ]:
# Your code here

### Solution

In [ ]:
%%writefile train_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from common import make_dataset, make_model

dist.init_process_group("gloo")
rank = dist.get_rank()
world_size = dist.get_world_size()

dataset = make_dataset()
model = DDP(make_model())
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

for epoch in range(3):
  sampler.set_epoch(epoch)
  for x, y in loader:
    optimizer.zero_grad()
    loss = cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
  if rank == 0:
    print(f"Epoch {epoch}: last loss {loss.item():.3f}")

if rank == 0:
  torch.save(model.module.state_dict(), "model.pt")
  print("Checkpoint saved")

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 train_ddp.py

In [ ]:
import torch

from common import make_model

reloaded = make_model()
reloaded.load_state_dict(torch.load("model.pt"))
print("Weights reloaded into a regular model")

`model.state_dict()` would prefix every key with `module.` (the name of the attribute where DDP stores the model): these weights couldn't be reloaded into a regular model without renaming the keys.

## With PyTorch Lightning

Lightning handles everything we wrote by hand: process group, `DDP`, `DistributedSampler`, `set_epoch`, printing and saving from rank 0. You just pick a strategy in the `Trainer`.

*Complete the `Trainer` of `train_lightning.py` to train on 2 CPU processes with the `"ddp"` strategy, then launch the script. On a machine with 4 GPUs, what would need to change?*

In [ ]:
%%writefile train_lightning.py
import lightning as L
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model


class Classifier(L.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.model = make_model()

  def training_step(self, batch, batch_idx):
    x, y = batch
    loss = cross_entropy(self.model(x), y)
    self.log("train_loss", loss, prog_bar=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  loader = DataLoader(make_dataset(), batch_size=64, shuffle=True)
  trainer = L.Trainer(max_epochs=3, ...)  # Your code here
  trainer.fit(Classifier(), loader)

### Solution

In [ ]:
%%writefile train_lightning.py
import lightning as L
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from common import make_dataset, make_model


class Classifier(L.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.model = make_model()

  def training_step(self, batch, batch_idx):
    x, y = batch
    loss = cross_entropy(self.model(x), y)
    self.log("train_loss", loss, prog_bar=True)
    return loss

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  loader = DataLoader(make_dataset(), batch_size=64, shuffle=True)
  trainer = L.Trainer(max_epochs=3, accelerator="cpu", devices=2, strategy="ddp",
                      enable_checkpointing=False, logger=False)
  trainer.fit(Classifier(), loader)

In [ ]:
!python train_lightning.py

On 4 GPUs, writing `accelerator="gpu", devices=4` would be enough (plus `precision="bf16-mixed"` for mixed precision). Lightning itself replaces the `DataLoader`'s `shuffle=True` with a `DistributedSampler`.

## To go further

- If the model doesn't fit in one GPU's memory, DDP is no longer enough: the `"fsdp"` strategy (*Fully Sharded Data Parallel*) also shards weights, gradients and optimizer states across GPUs.
- On several machines, `torchrun` also takes `--nnodes`, `--node_rank` and the address of a master machine (`--rdzv_endpoint`).
- See [PyTorch's DDP tutorial](https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html) and the [`torchrun` documentation](https://docs.pytorch.org/docs/stable/elastic/run.html).